<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=341877854" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 1 SETUP + VERIFY =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, MaxPooling2D,
                                      Dropout, GlobalAveragePooling2D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 1
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

# --- load the known, confirmed fold assignment file directly ---
CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch, this is not the verified fold file"

print(f"\nFold sizes:\n{cv_assignments['fold'].value_counts().sort_index()}")
coverage = pd.crosstab(cv_assignments['fold'], cv_assignments['label'])
assert (coverage[['df','vasc']] > 0).all().all(), "STOP: rare class missing from a fold"
print("Rare class coverage check: PASS")

# ---------- fold split builder, fillna bug already patched ----------
def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"Proportions: train {len(train_df)/len(cv_assignments)*100:.1f}% | val {len(val_df)/len(cv_assignments)*100:.1f}% | test {len(test_df)/len(cv_assignments)*100:.1f}%")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    "STOP: FOLD 1 LEAKAGE detected"
print("Leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

def make_fold_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Ready to train. =====")

2026-08-12 08:16:52.660019: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786522612.870368      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786522612.934889      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786522613.438565      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786522613.438603      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786522613.438605      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

Fold sizes:
fold
1    4833
2    4868
3    4754
4    4683
5    4698
Name: count, dtype: int64
Rare class coverage check: PASS

===== FOLD 1 SPLIT =====
Train 16,121 | Val 2,882 | Test 4,833
Proportions: train 67.6% | val 12.1% | test 20.3%
Leakage check: PASS
Fold 1 class_weight: {np.str_('bcc'): np.float64(1.184), np.str_('bkl'): np.float64(1.514), np.str_('df'): np.float64(16.284), np.str_('melanoma'): np.float64(0.88), np.str_('nevus'): np.float64(0.309), np.str_('vasc'): np.float64(15.442)}

===== Fold 1 setup verified. Ready to train. =====


In [2]:
# ===== STAGE 16, FOLD 1: TRAIN CUSTOM CNN + EFFICIENTNETB0 =====
fold_results = []

# ---------- CUSTOM CNN ----------
try:
    print(f"{'='*60}\nFOLD {CURRENT_FOLD}: Custom CNN\n{'='*60}")
    tr, va, te = make_fold_gens(None)
    print("class_indices:", te.class_indices)

    model = build_custom_cnn()
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom_log.csv', append=False)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_custom.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    fold_results.append(dict(fold=CURRENT_FOLD, arch='custom', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true)))
    print(f"RESULT: {fold_results[-1]}")
    del model; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} Custom CNN FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

# ---------- EFFICIENTNETB0 ----------
try:
    print(f"\n{'='*60}\nFOLD {CURRENT_FOLD}: EfficientNetB0\n{'='*60}")
    tr, va, te = make_fold_gens(eff_pre)
    print("class_indices:", te.class_indices)

    model, base = build_pretrained(EfficientNetB0)
    log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_log.csv'

    base.trainable = False
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
    print("Phase 1 sanity:", model.evaluate(te, verbose=0))

    base.trainable = True
    model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(log_file, append=True)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_eff.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    fold_results.append(dict(fold=CURRENT_FOLD, arch='eff', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true)))
    print(f"RESULT: {fold_results[-1]}")
    del model, base; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} EfficientNetB0 FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

# ---------- SAVE ----------
results_path = f'/kaggle/working/cv_f{CURRENT_FOLD}_results.csv'
pd.DataFrame(fold_results).to_csv(results_path, index=False)

print(f"\n{'='*60}\nFOLD {CURRENT_FOLD} PARTIAL SUMMARY (custom + eff)\n{'='*60}")
for r in fold_results:
    print(f"{r['arch']:8} acc={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}  macro_auc={r['macro_auc']:.4f}")
print(f"\nCompleted this run: {len(fold_results)}/2. Still needed for fold {CURRENT_FOLD}: mob, res.")
print(f"Saved: {results_path}")

FOLD 1: Custom CNN
Found 16121 validated image filenames belonging to 6 classes.
Found 2882 validated image filenames belonging to 6 classes.
Found 4833 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1786522689.667014      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786522689.672936      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/60


E0000 00:00:1786522693.047769      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1786522694.428149      66 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786522696.805923      66 service.cc:152] XLA service 0x799079c09b40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786522696.805957      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786522696.805961      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786522697.071528      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


504/504 [==============================] - 699s 1s/step - loss: 1.7665 - accuracy: 0.2808 - val_loss: 2.2152 - val_accuracy: 0.0763
Epoch 2/60
504/504 [==============================] - 461s 915ms/step - loss: 1.6344 - accuracy: 0.3324 - val_loss: 1.6347 - val_accuracy: 0.2949
Epoch 3/60
504/504 [==============================] - 455s 903ms/step - loss: 1.5579 - accuracy: 0.3570 - val_loss: 1.6805 - val_accuracy: 0.2911
Epoch 4/60
504/504 [==============================] - 463s 919ms/step - loss: 1.4837 - accuracy: 0.3933 - val_loss: 1.3979 - val_accuracy: 0.4171
Epoch 5/60
504/504 [==============================] - 462s 916ms/step - loss: 1.4983 - accuracy: 0.3919 - val_loss: 1.2933 - val_accuracy: 0.4663
Epoch 6/60
504/504 [==============================] - 464s 920ms/step - loss: 1.4712 - accuracy: 0.4075 - val_loss: 1.5202 - val_accuracy: 0.3428
Epoch 7/60
504/504 [==============================] - 458s 909ms/step - loss: 1.4494 - accuracy: 0.4192 - val_loss: 1.2585 - val_accuracy:

E0000 00:00:1786529557.914802      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


504/504 [==============================] - 357s 696ms/step - loss: 1.3584 - accuracy: 0.4579 - val_loss: 1.0166 - val_accuracy: 0.6083
Epoch 2/10
504/504 [==============================] - 358s 710ms/step - loss: 1.0915 - accuracy: 0.5444 - val_loss: 1.1984 - val_accuracy: 0.5146
Epoch 3/10
504/504 [==============================] - 353s 701ms/step - loss: 0.9872 - accuracy: 0.5634 - val_loss: 1.0764 - val_accuracy: 0.5829
Epoch 4/10
504/504 [==============================] - 359s 712ms/step - loss: 0.9614 - accuracy: 0.5719 - val_loss: 1.0311 - val_accuracy: 0.5802
Epoch 5/10
504/504 [==============================] - 354s 702ms/step - loss: 0.8785 - accuracy: 0.6007 - val_loss: 1.0967 - val_accuracy: 0.5833
Epoch 6/10
504/504 [==============================] - 365s 724ms/step - loss: 0.8795 - accuracy: 0.6038 - val_loss: 0.9272 - val_accuracy: 0.6235
Epoch 7/10
504/504 [==============================] - 385s 763ms/step - loss: 0.8344 - accuracy: 0.6116 - val_loss: 1.1279 - val_accura

E0000 00:00:1786533351.210288      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


504/504 [==============================] - 552s 998ms/step - loss: 3.8628 - accuracy: 0.4066 - val_loss: 1.5519 - val_accuracy: 0.4885
Epoch 2/60
504/504 [==============================] - 494s 980ms/step - loss: 1.8761 - accuracy: 0.4432 - val_loss: 1.4735 - val_accuracy: 0.4951
Epoch 3/60
504/504 [==============================] - 469s 931ms/step - loss: 1.4221 - accuracy: 0.4699 - val_loss: 1.3889 - val_accuracy: 0.5115
Epoch 4/60
504/504 [==============================] - 456s 905ms/step - loss: 1.1949 - accuracy: 0.4965 - val_loss: 1.2753 - val_accuracy: 0.5371
Epoch 5/60
504/504 [==============================] - 459s 910ms/step - loss: 1.0960 - accuracy: 0.5240 - val_loss: 1.1667 - val_accuracy: 0.5760
Epoch 6/60
504/504 [==============================] - 460s 912ms/step - loss: 1.0211 - accuracy: 0.5487 - val_loss: 1.1275 - val_accuracy: 0.5843
Epoch 7/60
504/504 [==============================] - 459s 911ms/step - loss: 0.9444 - accuracy: 0.5677 - val_loss: 1.0666 - val_accura